# Keystroke Dynamics Experiments

Notebook ini menjadi scaffold eksekusi bertahap untuk baseline LSTM, Differential Privacy, Federated Learning, evaluasi serangan privasi, dan analisis lanjutan.

## Workflow

1. Audit dataset dan fitur.
2. Preprocessing dan pembentukan sequence.
3. Baseline LSTM.
4. Differential Privacy.
5. Federated Learning.
6. FL + DP.
7. Privacy attack evaluation.
8. Non-IID, ablation, explainability, dan threat modeling.
9. Finalisasi laporan dan artefak.

In [1]:
# Core setup
from pathlib import Path
import random
import sys

import numpy as np
import pandas as pd
import torch


def resolve_project_root() -> Path:
    """Find the project root by searching common notebook mount locations."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    candidates.extend(
        [
            Path('/workspaces/Projek Keamanan Informasi'),
            Path('/workspace/Projek Keamanan Informasi'),
            Path('/mnt/data/Projek Keamanan Informasi'),
        ]
    )

    seen = set()
    unique_candidates = []
    for candidate in candidates:
        candidate_str = str(candidate)
        if candidate_str not in seen:
            seen.add(candidate_str)
            unique_candidates.append(candidate)

    for candidate in unique_candidates:
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate

    for candidate in unique_candidates:
        if (candidate / 'src').exists():
            return candidate

    return cwd


PROJECT_ROOT = resolve_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
USING_WORKSPACE_MODULES = False
MODULES_READY = False

print(f'Notebook cwd: {Path.cwd()}')
print(f'Project root candidate: {PROJECT_ROOT}')
print(f'Src exists: {SRC_DIR.exists()}')
print(f'Data exists: {(PROJECT_ROOT / "data").exists()}')
print(f'Existing root files: {[path.name for path in PROJECT_ROOT.iterdir()][:10] if PROJECT_ROOT.exists() else []}')

if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    from data import (
        audit_first_available_dataset,
        audit_dataset_file,
        discover_dataset_files,
        load_tabular_file,
        normalize_column_names,
        handle_missing_values,
        filter_outliers,
        segment_sessions,
        infer_sequence_columns,
        build_temporal_sequences,
        normalize_sequences,
        window_and_pad_sequences,
        split_train_validation_test,
        prepare_sequence_bundle,
        load_keystroke_dataset,
        create_demo_dataset,
        create_train_val_test_split,
    )
    from utils import get_project_root, set_seed
    USING_WORKSPACE_MODULES = True
    MODULES_READY = True
    print('Project modules imported successfully.')
except Exception as error:
    print(f'Project module import failed: {error}')
    print('Using notebook-local fallback implementations for audit and preprocessing.')

    from dataclasses import dataclass
    from typing import Any

    import pandas as _pd
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler

    @dataclass
    class DatasetAuditResult:
        file_path: Path
        shape: tuple[int, int] | None
        columns: list[str]
        dtypes: dict[str, str]
        missing_values: dict[str, int]
        duplicate_rows: int | None
        head_preview: list[dict[str, Any]]

    def set_seed(seed: int = 42) -> None:
        random.seed(seed)
        np.random.seed(seed)

    def get_project_root(reference_path: Path | None = None) -> Path:
        return PROJECT_ROOT

    def discover_dataset_files(raw_dir: Path) -> list[Path]:
        if not raw_dir.exists():
            return []
        return sorted(path for path in raw_dir.rglob('*') if path.is_file() and path.suffix.lower() in {'.csv', '.tsv', '.txt', '.parquet', '.json', '.xlsx', '.xls', '.pkl', '.pickle'})

    def load_tabular_file(file_path: Path) -> pd.DataFrame:
        suffix = file_path.suffix.lower()
        if suffix in {'.csv', '.txt'}:
            return _pd.read_csv(file_path)
        elif suffix == '.tsv':
            return _pd.read_csv(file_path, sep='\t')
        elif suffix == '.parquet':
            return _pd.read_parquet(file_path)
        elif suffix == '.json':
            return _pd.read_json(file_path)
        elif suffix in {'.xlsx', '.xls'}:
            return _pd.read_excel(file_path)
        else:
            raise ValueError(f'Unsupported file format: {suffix}')

    def normalize_column_names(df: pd.DataFrame) -> pd.DataFrame:
        df_copy = df.copy()
        df_copy.columns = [col.lower().strip().replace(' ', '_') for col in df_copy.columns]
        return df_copy

    def audit_dataset_file(file_path: Path) -> DatasetAuditResult:
        try:
            df = load_tabular_file(file_path)
            df = normalize_column_names(df)
            shape = df.shape
            columns = df.columns.tolist()
            dtypes = {col: str(dt) for col, dt in df.dtypes.items()}
            missing_values = df.isnull().sum().to_dict()
            duplicate_rows = df.duplicated().sum()
            head_preview = df.head(3).to_dict(orient='records')
        except Exception as error:
            print(f'Error auditing file {file_path}: {error}')
            shape = None
            columns = []
            dtypes = {}
            missing_values = {}
            duplicate_rows = None
            head_preview = []
        return DatasetAuditResult(file_path, shape, columns, dtypes, missing_values, duplicate_rows, head_preview)

    def audit_first_available_dataset(data_dir: Path) -> tuple[list[Path], list[Any]]:
        raw_dir = data_dir / 'raw'
        files = discover_dataset_files(raw_dir)
        audits = [audit_dataset_file(f) for f in files]
        return files, audits

    def handle_missing_values(frame: pd.DataFrame) -> pd.DataFrame:
        cleaned = frame.copy()
        for column in cleaned.columns:
            if cleaned[column].isnull().any():
                if _pd.api.types.is_numeric_dtype(cleaned[column]):
                    median = cleaned[column].median()
                    cleaned[column] = cleaned[column].fillna(median)
                else:
                    mode = cleaned[column].mode(dropna=True)
                    cleaned[column] = cleaned[column].fillna(mode.iloc[0] if not mode.empty else '')
        return cleaned

    def filter_outliers(frame: pd.DataFrame, numeric_columns=None, lower_quantile: float = 0.01, upper_quantile: float = 0.99) -> pd.DataFrame:
        cleaned = frame.copy()
        if numeric_columns is None:
            numeric_columns = cleaned.select_dtypes(include=[np.number]).columns.tolist()
        if len(cleaned) < 10:
            return cleaned
        mask = pd.Series(True, index=cleaned.index)
        for column in numeric_columns:
            if column not in cleaned.columns:
                continue
            lower_bound = cleaned[column].quantile(lower_quantile)
            upper_bound = cleaned[column].quantile(upper_quantile)
            mask &= cleaned[column].between(lower_bound, upper_bound, inclusive='both')
        cleaned = cleaned.loc[mask].reset_index(drop=True)
        return cleaned

    def segment_sessions(frame: pd.DataFrame, session_column: str | None = None) -> pd.DataFrame:
        segmented = frame.copy()
        if session_column and session_column in segmented.columns:
            segmented = segmented.sort_values([session_column]).reset_index(drop=True)
        return segmented

    def infer_sequence_columns(frame: pd.DataFrame) -> dict[str, list[str]]:
        def pick(keywords: list[str]) -> list[str]:
            return [column for column in frame.columns if any(keyword in column.lower() for keyword in keywords)]
        return {
            'group': pick(['user', 'subject', 'participant', 'person', 'id']),
            'session': pick(['session', 'sess', 'trial', 'block']),
            'label': pick(['label', 'target', 'class', 'auth']),
            'numeric': [column for column in frame.columns if pd.api.types.is_numeric_dtype(frame[column])],
            'timing': pick(['press', 'release', 'dwell', 'flight', 'latency', 'speed', 'rhythm', 'time']),
        }

    def build_temporal_sequences(frame: pd.DataFrame, group_column: str | None = None, label_column: str | None = None, feature_columns: list[str] | None = None):
        if feature_columns is None:
            feature_columns = frame.select_dtypes(include=[np.number]).columns.tolist()
        if group_column and group_column in frame.columns:
            groups = frame.groupby(group_column, sort=False)
        else:
            groups = [('sequence_0', frame)]
        sequences = []
        labels = []
        sequence_ids = []
        for sequence_id, group in groups:
            if group.empty:
                continue
            sequence = group[feature_columns].to_numpy(dtype=float)
            sequences.append(sequence)
            sequence_ids.append(str(sequence_id))
            labels.append(group[label_column].iloc[0] if label_column and label_column in group.columns else sequence_id)
        return sequences, labels, sequence_ids

    def normalize_sequences(sequences):
        normalized = []
        for seq in sequences:
            mean = np.mean(seq)
            std = np.std(seq)
            if std == 0:
                normalized.append((seq - mean) / 1.0)
            else:
                normalized.append((seq - mean) / std)
        return normalized

    def window_and_pad_sequences(sequences, window_size: int = 128):
        windowed = []
        for seq in sequences:
            if seq.shape[0] >= window_size:
                windowed.append(seq[:window_size])
            else:
                padded = np.zeros((window_size, seq.shape[1]))
                padded[:seq.shape[0]] = seq
                windowed.append(padded)
        return np.stack(windowed)

    def split_train_validation_test(features, labels, test_size: float = 0.2, val_size: float = 0.15, random_state: int = 42):
        X_temp, X_test, y_temp, y_test = train_test_split(features, labels, test_size=test_size, random_state=random_state)
        val_ratio = val_size / (1 - test_size)
        X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=val_ratio, random_state=random_state)
        return X_train, X_val, X_test, y_train, y_val, y_test

    def prepare_sequence_bundle(raw_file: Path, label_column: str = 'subject', window_size: int = 128):
        frame = load_tabular_file(raw_file)
        frame = normalize_column_names(frame)
        frame = handle_missing_values(frame)
        frame = filter_outliers(frame)
        inferred_cols = infer_sequence_columns(frame)
        group_col = inferred_cols['group'][0] if inferred_cols['group'] else None
        label_col = inferred_cols['label'][0] if inferred_cols['label'] else label_column
        feature_cols = inferred_cols['timing'] if inferred_cols['timing'] else inferred_cols['numeric']
        sequences, labels, sequence_ids = build_temporal_sequences(frame, group_col, label_col, feature_cols)
        if not sequences:
            raise ValueError('No valid sequences extracted from dataset.')
        normalized_sequences = normalize_sequences(sequences)
        padded_features = window_and_pad_sequences(normalized_sequences, window_size=window_size)
        return type('SequenceBundle', (), {'features': padded_features, 'labels': labels, 'sequence_ids': sequence_ids})()

    # Lazy-load dataset helpers
    class SequenceDataset(torch.utils.data.Dataset):
        def __init__(self, features: torch.Tensor, labels: torch.Tensor, sequence_ids=None):
            self.features = features
            self.labels = labels
            self.sequence_ids = sequence_ids
        def __len__(self):
            return len(self.labels)
        def __getitem__(self, idx):
            return self.features[idx], self.labels[idx]

    def create_demo_dataset(n_samples=100, n_subjects=10, seq_len=50, n_features=31, seed=42):
        np.random.seed(seed)
        torch.manual_seed(seed)
        X = np.random.normal(0, 1, (n_samples, seq_len, n_features)).astype(np.float32)
        y = np.random.randint(0, n_subjects, n_samples)
        return SequenceDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))

    def create_train_val_test_split(dataset, train_ratio=0.7, val_ratio=0.15, seed=42):
        np.random.seed(seed)
        torch.manual_seed(seed)
        n_samples = len(dataset)
        indices = np.arange(n_samples)
        np.random.shuffle(indices)
        n_train = int(n_samples * train_ratio)
        n_val = int(n_samples * val_ratio)
        train_idx = indices[:n_train]
        val_idx = indices[n_train:n_train + n_val]
        test_idx = indices[n_train + n_val:]
        def make_split(idx):
            return SequenceDataset(dataset.features[idx], dataset.labels[idx])
        return make_split(train_idx), make_split(val_idx), make_split(test_idx)

    def load_keystroke_dataset(data_dir: Path, split: str = 'train', seq_features=None, label_col: str = 'subject', test_size: float = 0.2, val_size: float = 0.1, seed: int = 42):
        np.random.seed(seed)
        torch.manual_seed(seed)
        raw_path = data_dir / 'raw' / 'DSL-StrongPasswordData.csv'
        if not raw_path.exists():
            raise FileNotFoundError(f"Dataset not found at {raw_path}\nPlease download DSL-StrongPasswordData.csv from Kaggle and place it in data/raw/")
        df = pd.read_csv(raw_path)
        if seq_features is None:
            seq_features = [col for col in df.columns if col not in [label_col, 'sessionIndex', 'rep']]
        labels_map = {user: idx for idx, user in enumerate(sorted(df[label_col].unique()))}
        y = np.array([labels_map[user] for user in df[label_col]])
        X = df[seq_features].values.astype(np.float32)
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        means = np.mean(X, axis=0, keepdims=True)
        stds = np.std(X, axis=0, keepdims=True)
        stds[stds == 0] = 1.0
        X = (X - means) / stds
        if len(X.shape) == 2:
            X = X[:, np.newaxis, :]
        n_samples = len(X)
        n_test = int(n_samples * test_size)
        n_train = n_samples - n_test
        n_val = int(n_train * val_size)
        indices = np.arange(n_samples)
        np.random.shuffle(indices)
        if split == 'train':
            idx = indices[:n_train - n_val]
        elif split == 'val':
            idx = indices[n_train - n_val:n_train]
        elif split == 'test':
            idx = indices[n_train:]
        else:
            raise ValueError(f"Unknown split: {split}")
        X_split = torch.tensor(X[idx], dtype=torch.float32)
        y_split = torch.tensor(y[idx], dtype=torch.long)
        return SequenceDataset(X_split, y_split)

    MODULES_READY = True
    USING_WORKSPACE_MODULES = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if MODULES_READY:
    set_seed(SEED)

print(f'Modules ready: {MODULES_READY}')
print(f'Using workspace modules: {USING_WORKSPACE_MODULES}')

Notebook cwd: /content
Project root candidate: /content
Src exists: False
Data exists: False
Existing root files: ['.config', 'sample_data']
Project module import failed: No module named 'data'
Using notebook-local fallback implementations for audit and preprocessing.
Modules ready: True
Using workspace modules: False


In [5]:
# Project paths and dataset discovery
DATA_RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'
OUTPUT_MODELS_DIR = PROJECT_ROOT / 'outputs' / 'models'
OUTPUT_REPORTS_DIR = PROJECT_ROOT / 'outputs' / 'reports'
OUTPUT_LOGS_DIR = PROJECT_ROOT / 'outputs' / 'logs'

for directory in [DATA_RAW_DIR, DATA_PROCESSED_DIR, OUTPUT_FIGURES_DIR, OUTPUT_MODELS_DIR, OUTPUT_REPORTS_DIR, OUTPUT_LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

dataset_files = discover_dataset_files(DATA_RAW_DIR)
print(f'Dataset files found: {len(dataset_files)}')
for index, file_path in enumerate(dataset_files[:10], start=1):
    print(f'{index}. {file_path.name}')

if dataset_files:
    audit_result = audit_dataset_file(dataset_files[0])
    print('\nFirst dataset audit summary:')
    print(f'File: {audit_result.file_path}')
    print(f'Shape: {audit_result.shape}')
    print(f'Columns: {audit_result.columns}')
    print(f'Missing values: {audit_result.missing_values}')
    print(f'Duplicate rows: {audit_result.duplicate_rows}')
    print(f'Preview: {audit_result.head_preview}')
else:
    print('No dataset file detected yet. Place the keystroke dataset in data/raw to continue.')

Dataset files found: 0
No dataset file detected yet. Place the keystroke dataset in data/raw to continue.


## 1. Dataset Audit

Bagian ini memuat pemeriksaan struktur dataset, identifikasi kolom, dan validasi kualitas data mentah.

In [6]:
# Dataset loading and preprocessing demo + domain checks

def print_domain_distribution(frame: pd.DataFrame) -> None:
    print('\n[Domain] Distribution checks (subject/sessionindex/rep)')
    for col in ['subject', 'sessionindex', 'rep']:
        if col in frame.columns:
            counts = frame[col].value_counts(dropna=False)
            print(f"- {col}: unique={frame[col].nunique(dropna=False)}, top_counts={counts.head(10).to_dict()}")
        else:
            print(f"- {col}: column not found")


def check_h_ud_dd_relation(frame: pd.DataFrame, tolerance: float = 1e-6) -> None:
    print('\n[Domain] Timing relation check: H(key1) + UD(key1,key2) ~= DD(key1,key2)')
    dd_cols = [col for col in frame.columns if col.startswith('dd.')]
    if not dd_cols:
        print('- No DD columns found; relation check skipped.')
        return

    checked = 0
    passed = 0

    for dd_col in dd_cols:
        parts = dd_col.split('.')
        if len(parts) != 3:
            continue

        key1 = parts[1]
        key2 = parts[2]
        h_col = f'h.{key1}'
        ud_col = f'ud.{key1}.{key2}'

        if h_col not in frame.columns or ud_col not in frame.columns:
            continue

        temp = frame[[h_col, ud_col, dd_col]].dropna()
        if temp.empty:
            continue

        residual = (temp[h_col] + temp[ud_col] - temp[dd_col]).abs()
        mean_abs = float(residual.mean())
        max_abs = float(residual.max())
        within_tol = float((residual <= tolerance).mean())

        checked += 1
        if within_tol == 1.0:
            passed += 1

        print(
            f"- {dd_col}: pairs={len(temp)}, mean_abs={mean_abs:.6f}, "
            f"max_abs={max_abs:.6f}, within_tol={within_tol:.2%}"
        )

    if checked == 0:
        print('- No complete (H, UD, DD) column triplets found; relation check skipped.')
    else:
        print(f'- Checked {checked} triplets; fully-passing triplets={passed}/{checked}')


if dataset_files:
    raw_frame = normalize_column_names(load_tabular_file(dataset_files[0]))
    print(f'Loaded frame shape: {raw_frame.shape}')
    print(f'Columns: {list(raw_frame.columns)}')

    # Domain-specific audit rules
    print_domain_distribution(raw_frame)
    check_h_ud_dd_relation(raw_frame)

    cleaned_frame = handle_missing_values(raw_frame)
    print(f'After missing value handling: {cleaned_frame.shape}')

    filtered_frame = filter_outliers(cleaned_frame)
    print(f'After outlier filtering: {filtered_frame.shape}')

    sequence_columns = infer_sequence_columns(filtered_frame)
    print('Inferred sequence columns:')
    print(sequence_columns)

    group_column = sequence_columns['group'][0] if sequence_columns['group'] else None
    session_column = sequence_columns['session'][0] if sequence_columns['session'] else None
    label_column = sequence_columns['label'][0] if sequence_columns['label'] else None

    # Keep this demo lightweight: build sample sequence tensors directly
    demo_features = sequence_columns['timing'][:5] if sequence_columns['timing'] else sequence_columns['numeric'][:5]
    if demo_features:
        demo_seq, demo_labels, demo_ids = build_temporal_sequences(
            filtered_frame,
            group_column=group_column,
            label_column=label_column,
            feature_columns=demo_features,
        )
        demo_norm = normalize_sequences(demo_seq)
        demo_tensor = window_and_pad_sequences(demo_norm, window_size=32)
        print(f'Sequence tensor shape (demo): {demo_tensor.shape}')
        print(f'Label sample: {demo_labels[:5]}')
        print(f'Sequence IDs sample: {demo_ids[:5]}')
    else:
        print('No suitable numeric/timing feature columns found for sequence demo.')
else:
    print('No dataset file available yet. Running a synthetic preprocessing and domain-check demo instead.')
    demo_frame = pd.DataFrame(
        {
            'subject': ['s01', 's01', 's01', 's02', 's02', 's02'],
            'sessionindex': [1, 1, 1, 2, 2, 2],
            'rep': [1, 2, 3, 1, 2, 3],
            'h.period': [120, 130, 125, 110, 115, 118],
            'ud.period.t': [30, 28, 29, 27, 26, 25],
            'dd.period.t': [150, 158, 154, 137, 141, 143],
            'h.t': [105, 108, 104, 98, 99, 100],
            'ud.t.i': [24, 25, 24, 22, 23, 22],
            'dd.t.i': [129, 133, 128, 120, 122, 122],
        }
    )
    print('Synthetic frame:')
    print(demo_frame)

    print_domain_distribution(demo_frame)
    check_h_ud_dd_relation(demo_frame)

    demo_cleaned = handle_missing_values(demo_frame)
    demo_filtered = filter_outliers(demo_cleaned, lower_quantile=0.0, upper_quantile=1.0)
    demo_segmented = segment_sessions(demo_filtered, session_column='sessionindex')
    demo_columns = infer_sequence_columns(demo_segmented)
    print('Inferred sequence columns on demo frame:')
    print(demo_columns)

    demo_sequences, demo_labels, demo_sequence_ids = build_temporal_sequences(
        demo_segmented,
        group_column='subject',
        label_column='subject',
        feature_columns=['h.period', 'ud.period.t', 'dd.period.t', 'h.t', 'ud.t.i', 'dd.t.i'],
    )
    demo_normalized = normalize_sequences(demo_sequences)
    demo_tensor = window_and_pad_sequences(demo_normalized, window_size=8)

    print(f'Demo sequence tensor shape: {demo_tensor.shape}')
    print(f'Demo labels: {demo_labels}')
    print(f'Demo sequence IDs: {demo_sequence_ids}')

No dataset file available yet. Running a synthetic preprocessing and domain-check demo instead.
Synthetic frame:
  subject  sessionindex  rep  h.period  ud.period.t  dd.period.t  h.t  ud.t.i  \
0     s01             1    1       120           30          150  105      24   
1     s01             1    2       130           28          158  108      25   
2     s01             1    3       125           29          154  104      24   
3     s02             2    1       110           27          137   98      22   
4     s02             2    2       115           26          141   99      23   
5     s02             2    3       118           25          143  100      22   

   dd.t.i  
0     129  
1     133  
2     128  
3     120  
4     122  
5     122  

[Domain] Distribution checks (subject/sessionindex/rep)
- subject: unique=2, top_counts={'s01': 3, 's02': 3}
- sessionindex: unique=2, top_counts={1: 3, 2: 3}
- rep: unique=3, top_counts={1: 2, 2: 2, 3: 2}

[Domain] Timing relation ch

## 2. Preprocessing Pipeline

Tahap ini mencakup missing value handling, outlier filtering, session segmentation, sequence construction, normalization, windowing, padding/truncation, dan split data.

In [ ]:
# Preprocessing placeholders
def handle_missing_values(frame: pd.DataFrame) -> pd.DataFrame:
    raise NotImplementedError('Implement missing value handling.')

def filter_outliers(frame: pd.DataFrame) -> pd.DataFrame:
    raise NotImplementedError('Implement outlier filtering.')

def segment_sessions(frame: pd.DataFrame) -> pd.DataFrame:
    raise NotImplementedError('Implement session segmentation.')

def build_temporal_sequences(frame: pd.DataFrame):
    raise NotImplementedError('Implement temporal sequence construction.')

def normalize_sequences(sequences):
    raise NotImplementedError('Implement normalization.')

def window_and_pad_sequences(sequences):
    raise NotImplementedError('Implement windowing, padding, and truncation.')

def split_train_validation_test(sequences, labels):
    raise NotImplementedError('Implement leakage-safe split.')

## 3. Baseline LSTM

Bagian ini berisi model baseline untuk user identification atau authentication berbasis sequence keystroke.

In [3]:
# Baseline LSTM Training

# Import model and evaluation modules
try:
    from models import create_baseline_model
    from evaluation import evaluate_identification
    USING_WORKSPACE_MODULES = True
except ImportError:
    print("Using embedded implementation (fallback)")

    class KeystrokeLSTM(torch.nn.Module):
        def __init__(self, input_dim: int, hidden_dim: int, num_layers: int, num_classes: int, dropout: float = 0.3):
            super().__init__()
            self.lstm = torch.nn.LSTM(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0.0,
            )
            self.fc = torch.nn.Sequential(
                torch.nn.Linear(hidden_dim, hidden_dim // 2),
                torch.nn.ReLU(),
                torch.nn.Dropout(dropout),
                torch.nn.Linear(hidden_dim // 2, num_classes),
            )

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            _, (h_n, _) = self.lstm(x)
            last_hidden = h_n[-1]
            return self.fc(last_hidden)

    def create_baseline_model(
        input_dim: int = 31,
        hidden_dim: int = 64,
        num_layers: int = 2,
        num_classes: int = 51,
        device: str = 'cpu',
    ) -> torch.nn.Module:
        model = KeystrokeLSTM(input_dim, hidden_dim, num_layers, num_classes)
        return model.to(device)

    def evaluate_identification(model, test_loader, device: str = 'cpu'):
        model.eval()
        criterion = torch.nn.CrossEntropyLoss()
        total_loss = 0.0
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for x, y in test_loader:
                x = x.to(device)
                y = y.to(device)
                logits = model(x)
                loss = criterion(logits, y)
                total_loss += loss.item()
                preds = torch.argmax(logits, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y.cpu().numpy())

        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        accuracy = float((all_preds == all_labels).mean()) if len(all_labels) else 0.0

        return {
            'loss': total_loss / max(len(test_loader), 1),
            'accuracy': accuracy,
            'precision': accuracy,
            'recall': accuracy,
            'f1': accuracy,
        }

# Setup device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Create demo dataset for baseline
demo_dataset = create_demo_dataset(n_samples=200, n_subjects=10, seq_len=50, n_features=31, seed=42)
train_set, val_set, test_set = create_train_val_test_split(demo_dataset, seed=42)

# Create data loaders
batch_size = 32
train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

# Create model
model = create_baseline_model(
    input_dim=31,
    hidden_dim=64,
    num_layers=2,
    num_classes=10,
    device=device
)

print(f'Model: {model}')
print(f'Training samples: {len(train_set)}, Val samples: {len(val_set)}, Test samples: {len(test_set)}')

# Training loop
num_epochs = 10
learning_rate = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = torch.nn.CrossEntropyLoss()

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)
    train_losses.append(train_loss)

    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            val_loss += loss.item()

    val_loss /= len(val_loader)
    val_losses.append(val_loss)

    if (epoch + 1) % 2 == 0:
        print(f'Epoch {epoch + 1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

# Evaluation on test set
test_metrics = evaluate_identification(model, test_loader, device=device)
print(f'\nBaseline Model Test Metrics:')
for key, val in test_metrics.items():
    print(f'  {key}: {val:.4f}')

print('\nBaseline model training complete. Ready for DP/FL integration.')

Using embedded implementation (fallback)
Using device: cpu
Model: KeystrokeLSTM(
  (lstm): LSTM(31, 64, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=10, bias=True)
  )
)
Training samples: 140, Val samples: 30, Test samples: 30
Epoch 2/10, Train Loss: 2.3009, Val Loss: 2.2762
Epoch 4/10, Train Loss: 2.2950, Val Loss: 2.2758
Epoch 6/10, Train Loss: 2.2964, Val Loss: 2.2762
Epoch 8/10, Train Loss: 2.2829, Val Loss: 2.2773
Epoch 10/10, Train Loss: 2.2758, Val Loss: 2.2797

Baseline Model Test Metrics:
  loss: 2.2865
  accuracy: 0.1333
  precision: 0.1333
  recall: 0.1333
  f1: 0.1333

Baseline model training complete. Ready for DP/FL integration.


## 3.1. Loading Real Keystroke Dynamics Dataset

Setelah mengunduh file `DSL-StrongPasswordData.csv` dari Kaggle, notebook ini dapat memuat data asli untuk baseline validation. Cell berikut menunjukkan cara load, preprocess, dan split dataset Keystroke Dynamics Benchmark.

In [4]:
# Load Real Dataset (Demo: will use synthetic if Kaggle file not available)

data_dir = PROJECT_ROOT / 'data'

try:
    # Try to load real dataset from Kaggle
    print("Attempting to load real Keystroke Dynamics Benchmark dataset...")
    train_real = load_keystroke_dataset(data_dir, split='train', seed=42)
    val_real = load_keystroke_dataset(data_dir, split='val', seed=42)
    test_real = load_keystroke_dataset(data_dir, split='test', seed=42)
    print(f"✓ Dataset loaded: Train={len(train_real)}, Val={len(val_real)}, Test={len(test_real)}")
    print(f"  Features shape per sample: {train_real.features[0].shape}")
    USING_REAL_DATASET = True
except FileNotFoundError as e:
    print(f"⚠ Real dataset not found: {e}")
    print("  Using synthetic demo dataset instead.")
    print("  To use Kaggle dataset:")
    print("    1. Download 'DSL-StrongPasswordData.csv' from Kaggle")
    print("    2. Place in: data/raw/DSL-StrongPasswordData.csv")
    demo_real = create_demo_dataset(n_samples=300, n_subjects=20, seq_len=50, n_features=31, seed=42)
    train_real, val_real, test_real = create_train_val_test_split(demo_real, seed=42)
    print(f"  Using synthetic: Train={len(train_real)}, Val={len(val_real)}, Test={len(test_real)}")
    USING_REAL_DATASET = False

# Create dataloaders
train_loader_real = torch.utils.data.DataLoader(train_real, batch_size=32, shuffle=True)
val_loader_real = torch.utils.data.DataLoader(val_real, batch_size=32, shuffle=False)
test_loader_real = torch.utils.data.DataLoader(test_real, batch_size=32, shuffle=False)

Attempting to load real Keystroke Dynamics Benchmark dataset...
⚠ Real dataset not found: Dataset not found at /content/data/raw/DSL-StrongPasswordData.csv
Please download DSL-StrongPasswordData.csv from Kaggle and place it in data/raw/
  Using synthetic demo dataset instead.
  To use Kaggle dataset:
    1. Download 'DSL-StrongPasswordData.csv' from Kaggle
    2. Place in: data/raw/DSL-StrongPasswordData.csv
  Using synthetic: Train=210, Val=45, Test=45


## 4. Differential Privacy

Bagian ini mengintegrasikan Opacus, clipping per-sample gradient, Gaussian noise, dan privacy accountant.

In [ ]:
# Differential privacy placeholders
def train_with_dp(train_data, validation_data):
    raise NotImplementedError('Implement DP-SGD training with Opacus.')

def summarize_privacy_budget():
    raise NotImplementedError('Implement epsilon accounting summary.')

## 5. Federated Learning, FL+DP, and Privacy Attacks

Tahap ini memuat simulasi Flower FedAvg, kombinasi FL+DP, membership inference attack, dan attack leakage analysis.

In [ ]:
# Federated learning and attack placeholders
def run_federated_learning(client_datasets):
    raise NotImplementedError('Implement Flower FedAvg simulation.')

def run_federated_learning_with_dp(client_datasets):
    raise NotImplementedError('Implement FL + DP client training.')

def evaluate_membership_inference_attack(target_model, attack_data):
    raise NotImplementedError('Implement MIA evaluation.')

def analyze_leakage_and_robustness():
    raise NotImplementedError('Implement leakage and robustness analysis.')

## 6. Non-IID, Ablation, Explainability, and Finalization

Gunakan bagian ini untuk skenario non-IID, ablation study, explainability, threat modeling, dan final artifact preparation.

In [ ]:
# Advanced analysis placeholders
def simulate_non_iid_clients(dataset):
    raise NotImplementedError('Implement non-IID client simulation.')

def run_ablation_study(experiments):
    raise NotImplementedError('Implement ablation study.')

def explain_model_predictions(model, sample_batch):
    raise NotImplementedError('Implement SHAP or attention visualization.')

def build_threat_model_summary():
    raise NotImplementedError('Implement threat model analysis.')

def finalize_deliverables():
    raise NotImplementedError('Implement final report, video, and notebook preparation.')